<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/phase9-portfolio-launch/03_conformity_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 9: Conformity Report
**Goal**: Build a general-purpose Conformity Report generator
      covering EU AI Act (11 articles), NIST AI RMF (4
      functions), NIST AI 600-1 (4 generative AI risk
      requirements), and ISO/IEC 42001 (3 clauses), 22
      obligations total. Untested obligations are honestly
      marked NOT ASSESSED rather than omitted. Includes an
      adapter for real Phase 8 agent results, where one piece
      of evidence often satisfies obligations across multiple
      frameworks at once, and a manual intake path for
      obligations with no automated audit behind them.
**Regulatory mapping**: EU AI Act Articles 9, 10, 11, 12, 13,
                    14, 15, 18, 26, 27, 99; NIST AI RMF Govern,
                    Map, Measure, Manage; NIST AI 600-1
                    hallucination tracking, source data lineage,
                    output monitoring, confabulation prevention;
                    ISO/IEC 42001 Clauses 6, 8, 9.
**Date**: June 2026.
**Status**: In Progress.

In [ ]:
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Any

# - EVIDENCE LEVELS (same discipline as the FRIA template) -
AUTOMATED = "AUTOMATED"
ATTESTED = "ATTESTED"
MISSING = "MISSING"

# - CONFORMITY STATUS -
COMPLIANT = "COMPLIANT"
NON_COMPLIANT = "NON_COMPLIANT"
PARTIAL = "PARTIAL"
NOT_ASSESSED = "NOT_ASSESSED"

# - THE LOCKED REGULATORY SCOPE, ACROSS THREE FRAMEWORKS -
# From the Afrispan Strategic Blueprint. EU AI Act gives 11 specific
# articles. NIST AI RMF gives 4 functions, not articles, since it is a
# risk-management methodology, not a list of legal obligations. ISO/IEC
# 42001 gives clauses, not articles, since it is a certifiable
# management-system standard. Each framework keeps its own native unit
# rather than being flattened into a single undifferentiated list, since
# a client needs to see which framework a given obligation belongs to.
REGULATORY_SCOPE = {
    "EU AI Act": {
        "Article 9": "Risk Management System",
        "Article 10": "Data Governance",
        "Article 11": "Technical Documentation",
        "Article 12": "Record-Keeping",
        "Article 13": "Transparency",
        "Article 14": "Human Oversight",
        "Article 15": "Accuracy, Robustness, Cybersecurity",
        "Article 18": "Quality Management System",
        "Article 26": "Deployer Obligations",
        "Article 27": "Fundamental Rights Impact Assessment",
        "Article 99": "Penalties",
    },
    "NIST AI RMF": {
        "GOVERN": "Organisational policies, culture, accountability for AI risk",
        "MAP": "Context and risk identification for the specific AI application",
        "MEASURE": "Quantitative and qualitative analysis of AI risks",
        "MANAGE": "Prioritising and acting on identified risks",
    },
    "NIST AI 600-1": {
        "Hallucination Tracking": "Detecting plausible but false model outputs",
        "Source Data Lineage": "Verifying outputs can be traced to source documents",
        "Output Monitoring": "Ongoing monitoring of generated content in production",
        "Confabulation Prevention": "Preventing fabricated sources and citations",
    },
    "ISO/IEC 42001": {
        "Clause 6": "Planning, AI risk assessment and treatment",
        "Clause 8": "Operation, AI system impact assessment and controls",
        "Clause 9": "Performance evaluation, monitoring and internal audit",
    },
}


@dataclass
class FrameworkConformity:
  """One obligation's conformity status, within one framework, with
  its evidence trail."""
  framework: str
  item: str
  obligation: str
  status: str = NOT_ASSESSED
  finding: str = ""
  evidence_level: str = MISSING
  source_reference: Optional[str] = None


@dataclass
class ConformityReportInput:
  """Generic Conformity Report input. Always covers the full locked
  scope across all three frameworks. Works for any audited system,
  any audit pipeline."""
  system_name: str
  system_purpose: str
  sector: str
  items: Dict[str, FrameworkConformity] = field(default_factory=dict)


def _scope_key(framework, item):
  return f"{framework} | {item}"


def _blank_scope() -> Dict[str, FrameworkConformity]:
  """Every report starts with every item, across every framework,
  present and NOT_ASSESSED, so nothing is silently missing."""
  items = {}
  for framework, obligations in REGULATORY_SCOPE.items():
    for item, obligation in obligations.items():
      key = _scope_key(framework, item)
      items[key] = FrameworkConformity(framework=framework, item=item, obligation=obligation)
  return items


# - ADAPTER: REAL PHASE 8 AGENT RESULTS TO CONFORMITY REPORT -
# One piece of Phase 8 evidence often satisfies obligations in more than
# one framework at once. The kill-switch and human authorization
# mechanism (Article 14) is also direct evidence for NIST's MANAGE
# function and, partially, ISO 42001's Clause 8 (operation, AI system
# controls). Each entry below lists every (framework, item) pair that
# a given agent's evidence speaks to, not just its EU AI Act article.
AGENT_SCOPE_MAP = {
    "Bias Audit Agent": [
        ("EU AI Act", "Article 10"),
        ("NIST AI RMF", "MEASURE"),
        ("ISO/IEC 42001", "Clause 6"),
    ],
    "Oversight Audit Agent": [
        ("EU AI Act", "Article 14"),
        ("NIST AI RMF", "MANAGE"),
        ("ISO/IEC 42001", "Clause 8"),
    ],
    "Documentation Audit Agent": [
        ("EU AI Act", "Article 11"),
        ("EU AI Act", "Article 18"),
        ("NIST AI RMF", "GOVERN"),
    ],
}

def from_agent_results(system, agent_results, audit_log_entries=None):
  """Build a ConformityReportInput from real Phase 8 agent_results.
  Optionally pass audit log entries so source_reference can point at a
  real hash instead of just the agent's own claim. If audit_log_entries
  is provided, Article 12 (Record-Keeping) is also populated directly
  from the log's own existence and integrity, since that obligation is
  evidenced by the log itself, not by any single agent's verdict."""
  system_name = system.get("system_name") or system.get("name", "Unknown System")
  items = _blank_scope()

  log_by_agent = {}
  if audit_log_entries:
    for entry in audit_log_entries:
      if entry["event_type"] == "AGENT_COMPLETED":
        log_by_agent[entry["agent"]] = entry

  for result in agent_results:
    agent_name = result.get("agent")
    scope_pairs = AGENT_SCOPE_MAP.get(agent_name, [])
    if not scope_pairs:
      continue

    verdict = result.get("verdict")
    status = COMPLIANT if verdict == "PASS" else NON_COMPLIANT if verdict == "FAIL" else PARTIAL

    finding_text = result.get("findings") or result.get("finding") or ""
    if isinstance(finding_text, list):
      formatted_findings = []
      for f in finding_text:
        if isinstance(f, dict):
          dim = f.get("dimension", "unknown dimension")
          ratio = f.get("ratio")
          severity = f.get("severity", "")
          formatted_findings.append(f"{dim.replace('_', ' ')}: ratio {ratio} ({severity})")
        else:
          formatted_findings.append(str(f))
      finding_text = "; ".join(formatted_findings)

    log_entry = log_by_agent.get(agent_name)
    source_ref = (f"audit_log entry #{log_entry['entry_number']}, "
                  f"hash {log_entry['entry_hash'][:12]}...") if log_entry else None

    for framework, item in scope_pairs:
      key = _scope_key(framework, item)
      if key not in items:
        continue
      items[key] = FrameworkConformity(
          framework=framework,
          item=item,
          obligation=REGULATORY_SCOPE[framework][item],
          status=status,
          finding=f"{agent_name} verdict: {verdict}. {finding_text}".strip(),
          evidence_level=AUTOMATED if log_entry else ATTESTED,
          source_reference=source_ref,
      )

  # Article 12, Record-Keeping, is evidenced by the audit log itself,
  # not by any single agent's verdict. If a real log was provided, its
  # existence and entry count are direct evidence of compliance.
  if audit_log_entries:
    key12 = _scope_key("EU AI Act", "Article 12")
    first_entry = audit_log_entries[0]
    last_entry = audit_log_entries[-1]
    items[key12] = FrameworkConformity(
        framework="EU AI Act",
        item="Article 12",
        obligation=REGULATORY_SCOPE["EU AI Act"]["Article 12"],
        status=COMPLIANT,
        finding=f"Immutable hash-chained audit log present, "
                f"{len(audit_log_entries)} entries recorded, "
                f"hash-chained from entry #{first_entry.get('entry_number', 1)} "
                f"to #{last_entry.get('entry_number', len(audit_log_entries))}.",
        evidence_level=AUTOMATED,
        source_reference=f"audit_log entry #{last_entry.get('entry_number')}, "
                          f"hash {last_entry.get('entry_hash', '')[:12]}...",
    )

  return ConformityReportInput(
      system_name=system_name,
      system_purpose=system.get("purpose", "Not specified"),
      sector=system.get("sector", "Not specified"),
      items=items,
  )


# - MANUAL INTAKE: ONE ITEM AT A TIME -
def intake_item(report_input, framework, item, status, finding, source):
  """Update one (framework, item) conformity status from a manually
  collected answer. Anything not called this way stays NOT_ASSESSED.

  Two safety rules, both load-bearing, not cosmetic:

  1. Never downgrade AUTOMATED evidence. If an item already carries
     AUTOMATED evidence (e.g. from a real Phase 8 audit log), a manual
     intake call is refused and the existing record is returned
     unchanged. Automated, tamper-evident evidence outranks manual
     attestation; it must never be silently overwritten by it.

  2. Never silently overwrite an existing ATTESTED finding. If the item
     already has a manual finding, the new finding is APPENDED to it
     (both kept, clearly separated), rather than one replacing the
     other. Two real findings about the same obligation, from two
     different sources, are both evidence; neither should disappear.
  """
  key = _scope_key(framework, item)
  finding = (finding or "").strip()
  source = (source or "").strip()

  existing = report_input.items.get(key)

  if existing and existing.evidence_level == AUTOMATED:
    print(f"  REFUSED: {key} already has AUTOMATED evidence. "
          f"Manual intake cannot downgrade it. No change made.")
    return report_input

  evidence_level = ATTESTED if (finding and source) else MISSING

  if existing and existing.evidence_level == ATTESTED and finding and source:
    combined_finding = f"{existing.finding}\n\n---\n\nAdditional finding: {finding}"
    combined_source = f"{existing.source_reference}; {source}"
    report_input.items[key] = FrameworkConformity(
        framework=framework,
        item=item,
        obligation=REGULATORY_SCOPE.get(framework, {}).get(item, "Unknown obligation"),
        status=status,
        finding=combined_finding,
        evidence_level=ATTESTED,
        source_reference=combined_source,
    )
    return report_input

  report_input.items[key] = FrameworkConformity(
      framework=framework,
      item=item,
      obligation=REGULATORY_SCOPE.get(framework, {}).get(item, "Unknown obligation"),
      status=status if (finding and source) else NOT_ASSESSED,
      finding=finding,
      evidence_level=evidence_level,
      source_reference=source if source else None,
  )
  return report_input


# - THE CONFORMITY REPORT GENERATOR -
def build_conformity_report(report_input):
  items = report_input.items

  status_counts = {COMPLIANT: 0, NON_COMPLIANT: 0, PARTIAL: 0, NOT_ASSESSED: 0}
  evidence_counts = {AUTOMATED: 0, ATTESTED: 0, MISSING: 0}
  for it in items.values():
    status_counts[it.status] += 1
    evidence_counts[it.evidence_level] += 1

  total = len(items)
  assessed = total - status_counts[NOT_ASSESSED]

  if status_counts[NON_COMPLIANT] > 0:
    overall = "NOT APPROVED, ONE OR MORE OBLIGATIONS NON-COMPLIANT"
  elif status_counts[PARTIAL] > 0:
    overall = "CONDITIONAL, ONE OR MORE OBLIGATIONS PARTIALLY COMPLIANT"
  elif assessed < total:
    overall = "INCOMPLETE ASSESSMENT, NOT ALL OBLIGATIONS ASSESSED"
  else:
    overall = "FULLY COMPLIANT"

  return {
      "system_name": report_input.system_name,
      "system_purpose": report_input.system_purpose,
      "sector": report_input.sector,
      "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
      "items": {k: asdict(v) for k, v in items.items()},
      "status_counts": status_counts,
      "evidence_counts": evidence_counts,
      "coverage_ratio": f"{assessed}/{total} obligations assessed",
      "automated_ratio": f"{evidence_counts[AUTOMATED]}/{total} obligations with automated evidence",
      "overall_verdict": overall,
  }


def render_conformity_markdown(report):
  status_badge = {
      COMPLIANT: "[COMPLIANT]",
      NON_COMPLIANT: "[NON-COMPLIANT]",
      PARTIAL: "[PARTIAL]",
      NOT_ASSESSED: "[NOT ASSESSED]",
  }
  evidence_badge = {
      AUTOMATED: "(automated evidence)",
      ATTESTED: "(manual attestation)",
      MISSING: "(no evidence)",
  }

  lines = [
      "# Conformity Report",
      f"**System:** {report['system_name']}",
      f"**Purpose:** {report['system_purpose']}",
      f"**Sector:** {report['sector']}",
      f"**Generated:** {report['generated_at']}",
      f"**Coverage:** {report['coverage_ratio']}",
      f"**Automated evidence:** {report['automated_ratio']}",
      "", "---", "",
  ]

  for framework, obligations in REGULATORY_SCOPE.items():
    lines.append(f"## {framework}")
    lines.append("")
    for item_key in obligations:
      key = _scope_key(framework, item_key)
      it = report["items"][key]
      badge = status_badge[it["status"]]
      ev_badge = evidence_badge[it["evidence_level"]]
      lines.append(f"### {it['item']}: {it['obligation']} {badge} {ev_badge}")
      if it["finding"]:
        lines.append(f"\n{it['finding']}")
      else:
        lines.append("\n*No finding recorded.*")
      if it["source_reference"]:
        lines.append(f"\n*Source: {it['source_reference']}*")
      lines.append("")
    lines.append("---")
    lines.append("")

  lines.append(f"**Overall verdict: {report['overall_verdict']}**")

  return "\n".join(lines)


# - DEMONSTRATION 1: FROM REAL PHASE 8 AGENT RESULTS -
print("====== DEMONSTRATION 1: CONFORMITY REPORT FROM REAL PHASE 8 RESULTS ======\n")

real_agent_results = [
    {"agent": "Bias Audit Agent", "article": "EU AI Act Article 10", "verdict": "FAIL",
     "critical_breach": True,
     "findings": [{"dimension": "gender_disparate_impact", "ratio": 0.43, "severity": "HIGH", "compliant": False},
                  {"dimension": "age_disparate_impact", "ratio": 0.20, "severity": "CRITICAL", "compliant": False},
                  {"dimension": "ethnicity_disparate_impact", "ratio": 0.80, "severity": "PASS", "compliant": True}]},
    {"agent": "Oversight Audit Agent", "article": "EU AI Act Article 14", "verdict": "FAIL",
     "critical_breach": True, "findings": "None currently implemented"},
    {"agent": "Documentation Audit Agent", "article": "EU AI Act Article 11 and 18", "verdict": "FAIL",
     "critical_breach": False, "finding": "Partial. No FRIA completed."},
]

real_audit_log = [
    {"entry_number": 1, "timestamp": "2026-06-16 14:43:20", "event_type": "AUDIT_STARTED",
     "agent": "Governance Controller", "details": "Auditing TalentMatch AI.Sector: employment.",
     "authorized_by": "Governance Controller", "previous_hash": "GENESIS",
     "entry_hash": "b3d6683215a6b11cab0a3409b2cfe14535c8bd230df47d01547577193769cd4e"},
    {"entry_number": 2, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "bias", "details": "Running bias audit agent.", "authorized_by": None,
     "previous_hash": "b3d6683215a6b11cab0a3409b2cfe14535c8bd230df47d01547577193769cd4e",
     "entry_hash": "073dc80ad6b6b49d4dde4f5c44509ec5893c9e99307b41c406f8a1c759206f33"},
    {"entry_number": 3, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Bias Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 10.",
     "authorized_by": None,
     "previous_hash": "073dc80ad6b6b49d4dde4f5c44509ec5893c9e99307b41c406f8a1c759206f33",
     "entry_hash": "8bae181de688928f6e58542af28f81bf80d3306b4c0256ad072e9e116e51fc23"},
    {"entry_number": 4, "timestamp": "2026-06-16 14:43:20", "event_type": "KILL_SWITCH_TRIGGERED",
     "agent": "Bias Audit Agent", "details": "Critical breach detected. EU AI Act Article 10.System halted immediately.",
     "authorized_by": None,
     "previous_hash": "8bae181de688928f6e58542af28f81bf80d3306b4c0256ad072e9e116e51fc23",
     "entry_hash": "10d1c68516911366fa9ddde0c0c3f0ed18ce5c1ccd6145d19deeb6c983429d3b"},
    {"entry_number": 5, "timestamp": "2026-06-16 14:43:20", "event_type": "HUMAN_AUTHORIZATION",
     "agent": "Human Auditor",
     "details": "position: Critical breach confirmed.Audit to continue under supervision pending full remediation plan.",
     "authorized_by": "Steve Onyeke, Lead AI Auditor, Afrispan Data Labs",
     "previous_hash": "10d1c68516911366fa9ddde0c0c3f0ed18ce5c1ccd6145d19deeb6c983429d3b",
     "entry_hash": "4eba89c9b220bc670510b6d4371fbcff9d3c4c1aae9e6be4e41ef6d37e88c143"},
    {"entry_number": 6, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "oversight", "details": "Running oversight audit agent.", "authorized_by": None,
     "previous_hash": "4eba89c9b220bc670510b6d4371fbcff9d3c4c1aae9e6be4e41ef6d37e88c143",
     "entry_hash": "2470f2d18845308a04b391255cbcb49bc8336f0d8d380f5a44578ddf3fb1245c"},
    {"entry_number": 7, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Oversight Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 14.",
     "authorized_by": None,
     "previous_hash": "2470f2d18845308a04b391255cbcb49bc8336f0d8d380f5a44578ddf3fb1245c",
     "entry_hash": "5e6a5f7b40598c304f52e20cf37c2abe4f90ec10cf0a6cb95476895dcbd26f00"},
    {"entry_number": 8, "timestamp": "2026-06-16 14:43:20", "event_type": "KILL_SWITCH_TRIGGERED",
     "agent": "Oversight Audit Agent", "details": "Critical breach detected. EU AI Act Article 14.System halted immediately.",
     "authorized_by": None,
     "previous_hash": "5e6a5f7b40598c304f52e20cf37c2abe4f90ec10cf0a6cb95476895dcbd26f00",
     "entry_hash": "18e78f8f01251a45034c79c30b7b39134addb0f0e775168e6cbe5405011b7613"},
    {"entry_number": 9, "timestamp": "2026-06-16 14:43:20", "event_type": "HUMAN_AUTHORIZATION",
     "agent": "Human Auditor",
     "details": "position: Critical breach confirmed.Audit to continue under supervision pending full remediation plan.",
     "authorized_by": "Steve Onyeke, Lead AI Auditor, Afrispan Data Labs",
     "previous_hash": "18e78f8f01251a45034c79c30b7b39134addb0f0e775168e6cbe5405011b7613",
     "entry_hash": "079c2dff6e1842792ac83e171691847496ac9c622c42bc78ab8b69668cc4150e"},
    {"entry_number": 10, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_STARTED",
     "agent": "documentation", "details": "Running documentation audit agent.", "authorized_by": None,
     "previous_hash": "079c2dff6e1842792ac83e171691847496ac9c622c42bc78ab8b69668cc4150e",
     "entry_hash": "8a7d7e0ae115de87c2846f43eea7e695346c0fd6275671bba6bbf45bdc3e78b1"},
    {"entry_number": 11, "timestamp": "2026-06-16 14:43:20", "event_type": "AGENT_COMPLETED",
     "agent": "Documentation Audit Agent", "details": "Verdict: FAIL.Article: EU AI Act Article 11 and 18.",
     "authorized_by": None,
     "previous_hash": "8a7d7e0ae115de87c2846f43eea7e695346c0fd6275671bba6bbf45bdc3e78b1",
     "entry_hash": "326a167657733472a825b76464240671ece18073c8bd2bac405d30efd8918d97"},
    {"entry_number": 12, "timestamp": "2026-06-16 14:43:20", "event_type": "AUDIT_COMPLETED",
     "agent": "Governance Controller",
     "details": "Audit completed. Agents run: 3. Agents halted: 0. Overall: CRITICAL NON-COMPLIANCE.",
     "authorized_by": None,
     "previous_hash": "326a167657733472a825b76464240671ece18073c8bd2bac405d30efd8918d97",
     "entry_hash": "7de55fa4f7f391fc52a874bdaee051ba76ad7171f0503661b78c20c764090a95"},
]

real_system = {
    "system_name": "TalentMatch AI",
    "purpose": "Automated CV screening",
    "sector": "employment",
}

report_input = from_agent_results(real_system, real_agent_results, real_audit_log)
report = build_conformity_report(report_input)
print(render_conformity_markdown(report))


# - DEMONSTRATION 2: MANUAL INTAKE ADDS AN UNASSESSED OBLIGATION -
print("\n\n====== DEMONSTRATION 2: MANUAL INTAKE ADDS ARTICLE 27 ======\n")

report_input = intake_item(
    report_input, "EU AI Act", "Article 27",
    status=PARTIAL,
    finding="FRIA generator built and verified against real audit data, "
            "but not yet run for every high-risk use case in production.",
    source="Phase 9 FRIA Template deliverable, verified 2026-06-25",
)

report2 = build_conformity_report(report_input)
print(render_conformity_markdown(report2))


# - DEMONSTRATION 3: MANUAL INTAKE FILLS IN NIST AI 600-1 FROM REAL PHASE 7 FINDINGS -
# TalentMatch AI (Phase 8's audit subject) has no RAG component, so the
# three Phase 8 agents correctly leave NIST AI 600-1 untouched. The real
# evidence for AI 600-1 lives in Phase 7's RAG pipeline findings instead,
# a different system, added here via manual intake rather than invented
# or borrowed from an unrelated audit.
print("\n\n====== DEMONSTRATION 3: NIST AI 600-1 FROM REAL PHASE 7 FINDINGS ======\n")

report_input = intake_item(
    report_input, "NIST AI 600-1", "Hallucination Tracking",
    status=COMPLIANT,
    finding="Grounded prompt restricts answers to retrieved sources; "
            "4 of 4 test queries returned grounded responses with no "
            "ungrounded claims introduced.",
    source="Phase 7 Lesson 1 RAG Pipeline findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Source Data Lineage",
    status=COMPLIANT,
    finding="Every RAG response includes the exact source documents "
            "used to generate the answer, enabling verification against "
            "original text.",
    source="Phase 7 Lesson 1 RAG Pipeline findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Output Monitoring",
    status=COMPLIANT,
    finding="Retrieval scoring logged per query; semantic intent check "
            "adds a second monitoring layer ahead of generation.",
    source="Phase 7 Lesson 3 Semantic Intent Check findings, June 2026",
)
report_input = intake_item(
    report_input, "NIST AI 600-1", "Confabulation Prevention",
    status=PARTIAL,
    finding="Model instructed to state when source documents do not "
            "contain the answer. Retrieval quality limitation found in "
            "testing (Article 5 vs Article 99 confusion under keyword "
            "scoring), since resolved by the Phase 9 Chroma migration.",
    source="Phase 7 Lesson 1 Finding 2, resolved per Phase 9 Section 1",
)

report3 = build_conformity_report(report_input)
print(render_conformity_markdown(report3))


# - DEMONSTRATION 4: PHASE 3-6 EVIDENCE, TESTING BOTH SAFETY RULES FOR REAL -
# Phase 4 (red-teaming) and Phase 6 (function-calling audit) both produced
# real findings relevant to Article 15 (robustness). Phase 5's bias
# finding is also real, but Article 10 already carries Phase 8's
# AUTOMATED evidence, so the safety rule below should refuse to let it
# overwrite that record, exactly as confirmed by direct testing.
print("\n\n====== DEMONSTRATION 4: PHASE 3-6 EVIDENCE ADDED ======\n")

# Phase 4: red-team findings map to Article 15 robustness
report_input = intake_item(
    report_input, "EU AI Act", "Article 15",
    status=NON_COMPLIANT,
    finding="Keyword-based jailbreak classifiers achieved only 28% "
            "accuracy. Translation-wrapper attacks bypassed safety "
            "classifiers consistently; academic/research framing was "
            "the highest-risk bypass vector identified.",
    source="Phase 4 Red-Teaming and Safety Analysis findings, June 2026",
)

# Phase 6: function-calling audit, a second real finding for the same
# article. Should APPEND to the Phase 4 finding above, not replace it.
report_input = intake_item(
    report_input, "EU AI Act", "Article 15",
    status=PARTIAL,
    finding="Tool selection accuracy 4/4 (100%) in testing, but the "
            "model silently mapped 'critical_catastrophic' severity to "
            "'prohibited' action without flagging the reclassification "
            "to a human reviewer.",
    source="Phase 6 Function Calling and Tool-Use Auditing findings, June 2026",
)

# Phase 5: bias findings. Article 10 already has Phase 8's AUTOMATED
# evidence, so this call should be REFUSED, not silently overwrite it.
print("Attempting to add Phase 5 bias evidence to Article 10 "
      "(already holds Phase 8 AUTOMATED evidence):")
report_input = intake_item(
    report_input, "EU AI Act", "Article 10",
    status=NON_COMPLIANT,
    finding="Age disparate impact ratio 0.20 (CRITICAL), gender "
            "disparate impact ratio 0.43 (FAILS 80% rule). Model used "
            "the word 'claimed' exclusively when describing younger "
            "candidates' experience, a qualitative disparity not "
            "captured by the ratio alone.",
    source="Phase 5 Bias Auditing and EU AI Act Compliance findings, June 2026",
)

# Phase 3: foundational evaluation methodology maps to NIST's MEASURE
report_input = intake_item(
    report_input, "NIST AI RMF", "MEASURE",
    status=PARTIAL,
    finding="Phase 3 established the evaluation methodology (accuracy, "
            "consistency, hallucination scoring) that later phases built "
            "on. Foundational, not a compliance finding in its own right.",
    source="Phase 3 LLM Evaluation findings, June 2026",
)

report4 = build_conformity_report(report_input)
print()
print(render_conformity_markdown(report4))


# - DEMONSTRATION 5: ARTICLE 99 FROM VERIFIED PROJECT HISTORY -
# Article 99 (Penalties) was left NOT_ASSESSED through Demo 4, which
# was a real gap, not a deliberate scoping choice. Checking the actual
# project transcripts directly (not memory, not assumption) confirmed
# substantive Article 99 work exists: the three-tier penalty structure
# was established, an LLM hallucinated an incorrect fine figure (30M
# euros / 6%), the error was caught via peer review, and the citation
# was corrected to the verified structure. This same correction event
# is independently confirmed in two separate transcripts and the Phase
# 5 study guide, giving genuine multi-source verification rather than
# a single unverified claim.
print("\n\n====== DEMONSTRATION 5: ARTICLE 99 FROM VERIFIED PROJECT HISTORY ======\n")

report_input = intake_item(
    report_input, "EU AI Act", "Article 99",
    status=COMPLIANT,
    finding="Three-tier penalty structure established and verified: "
            "Tier 1, Article 5 prohibited practices, up to 35 million "
            "euros or 7% global turnover. Tier 2, Article 99(3) "
            "high-risk non-compliance (Articles 10, 13, 14), up to 15 "
            "million euros or 3%. Tier 3, incorrect information to "
            "authorities, up to 7.5 million euros or 1%. An initial "
            "LLM-generated citation incorrectly stated 30 million "
            "euros or 6%; this was caught via peer review and "
            "corrected to the verified structure above.",
    source="Cross-verified across two independent project transcripts "
           "and the Phase 5 study guide, June 2026",
)

# This correction event is also itself evidence for NIST AI 600-1's
# Hallucination Tracking item, a second, independent instance of
# catching a fabricated claim, this time a legal fine figure rather
# than a RAG retrieval result. AUTOMATED evidence already exists there
# from Phase 7 (see Demonstration 3), so this is appended, not
# substituted, per the safety rules verified in Demonstration 4.
report_input = intake_item(
    report_input, "NIST AI 600-1", "Hallucination Tracking",
    status=COMPLIANT,
    finding="A second, independent hallucination was caught and "
            "corrected: an LLM-generated EU AI Act penalty citation "
            "(30 million euros or 6% turnover) was identified as "
            "incorrect via peer review and corrected to the verified "
            "three-tier structure (35M/7%, 15M/3%, 7.5M/1%).",
    source="Cross-verified across two independent project transcripts, "
           "June 2026",
)

report5 = build_conformity_report(report_input)
print(render_conformity_markdown(report5))

print("\n\n====== FINAL COMPARISON, ALL FIVE DEMONSTRATIONS ======")
print(f"Demo 1: {report['coverage_ratio']}, {report['overall_verdict']}")
print(f"Demo 2: {report2['coverage_ratio']}, {report2['overall_verdict']}")
print(f"Demo 3: {report3['coverage_ratio']}, {report3['overall_verdict']}")
print(f"Demo 4: {report4['coverage_ratio']}, {report4['overall_verdict']}")
print(f"Demo 5: {report5['coverage_ratio']}, {report5['overall_verdict']}")
print(f"\nArticle 10 evidence_level after Phase 5 intake attempt: "
      f"{report4['items'][_scope_key('EU AI Act', 'Article 10')]['evidence_level']} "
      f"(should still be AUTOMATED, unchanged from Phase 8)")

====== DEMONSTRATION 1: CONFORMITY REPORT FROM REAL PHASE 8 RESULTS ======

# Conformity Report
**System:** TalentMatch AI
**Purpose:** Automated CV screening
**Sector:** employment
**Generated:** 2026-06-25 21:20:51
**Coverage:** 10/22 obligations assessed
**Automated evidence:** 10/22 obligations with automated evidence

---

## EU AI Act

### Article 9: Risk Management System [NOT ASSESSED] (no evidence)

*No finding recorded.*

### Article 10: Data Governance [NON-COMPLIANT] (automated evidence)

Bias Audit Agent verdict: FAIL. gender disparate impact: ratio 0.43 (HIGH); age disparate impact: ratio 0.2 (CRITICAL); ethnicity disparate impact: ratio 0.8 (PASS)

*Source: audit_log entry #3, hash 8bae181de688...*

### Article 11: Technical Documentation [NON-COMPLIANT] (automated evidence)

Documentation Audit Agent verdict: FAIL. Partial. No FRIA completed.

*Source: audit_log entry #11, hash 326a16765773...*

### Article 12: Record-Keeping [COMPLIANT] (automated evidence)

Immutable 

## Findings: Conformity Report

**System audited:** TalentMatch AI (Automated CV screening, employment sector)
**Frameworks covered:** EU AI Act (11 articles), NIST AI RMF (4 functions),
                     NIST AI 600-1 (4 generative AI risk items), ISO/IEC
                     42001 (3 clauses). 22 obligations total, on every
                     report, regardless of how much evidence backs them.
**Demonstrations:** 5, run in sequence on the same growing record
**Date:** June 2026
**Regulatory mapping:** EU AI Act, NIST AI RMF, NIST AI 600-1, ISO/IEC 42001

---

### What Was Built

A Conformity Report generator that always declares its full 22-obligation
scope. An obligation with no test behind it is marked NOT ASSESSED, not
omitted, since a report that quietly drops untested obligations looks
more complete than it actually is.

The generator draws on real evidence from six phases of this curriculum,
not just the two most recent. Where multiple phases or multiple agents
produced findings relevant to the same obligation, all of them are
required to be visible, never one silently replacing another.

---

### Four Real Problems Found and Fixed During This Deliverable

**Problem 1, narrow scope.** The first working version only drew on
Phase 7 and Phase 8 evidence. Phases 3 through 6 had already produced
real, citable findings (Phase 4's red-team results, Phase 5's bias
audit, Phase 6's tool-use audit) that were sitting unused. Caught by
direct question, traced to a real cause: the most recent, most
structurally convenient evidence was used at the expense of the
project's actual breadth.

**Problem 2, silent data loss.** While correcting Problem 1, the
scenario needed to test it, two real findings landing on the same
obligation, exposed a defect in `intake_item()`: a new manual finding
would silently overwrite whatever was already recorded, including
AUTOMATED, tamper-evident evidence from a real audit log. Confirmed as
a live bug by direct test before any fix was written, confirmed
corrected by the same test afterward. Two safety rules were added:
AUTOMATED evidence cannot be downgraded by a manual intake call, and a
second ATTESTED finding is appended to an existing one rather than
replacing it.

---

### The Six Gaps Checked and Confirmed Genuinely Empty

| Gap | What exists in the project | Why it correctly stays NOT ASSESSED |
|---|---|---|
| Article 9 | Study-guide reference text only | No risk management system was ever built or tested |
| Article 13 | Study-guide reference text only | No transparency mechanism was ever built or tested |
| Article 26 | A conceptual blueprint positioning discussion | Conceptual understanding, not a tested finding |
| NIST MAP | Alignment-table explanation only | No risk-context-identification work was ever tested |
| ISO Clause 9 | LinkedIn/career-positioning mentions only | No performance-evaluation or internal-audit work was tested |

Each of these was checked directly against the real transcripts, the
same standard used to find Articles 12, 18, and 99. Confirming a gap is
genuinely empty is as important as filling one that is not: a report
that claims coverage it does not have is worse than one that honestly
shows where the evidence ends.

---

### The Full 22-Obligation Record, After All Five Demonstrations

| Framework | Item | Status | Evidence |
|---|---|---|---|
| EU AI Act | Article 9 | NOT ASSESSED | — |
| EU AI Act | Article 10 | NON-COMPLIANT | AUTOMATED |
| EU AI Act | Article 11 | NON-COMPLIANT | AUTOMATED |
| EU AI Act | Article 12 | COMPLIANT | AUTOMATED |
| EU AI Act | Article 13 | NOT ASSESSED | — |
| EU AI Act | Article 14 | NON-COMPLIANT | AUTOMATED |
| EU AI Act | Article 15 | PARTIAL | ATTESTED (Phase 4 + Phase 6, combined) |
| EU AI Act | Article 18 | NON-COMPLIANT | AUTOMATED |
| EU AI Act | Article 26 | NOT ASSESSED | — |
| EU AI Act | Article 27 | PARTIAL | ATTESTED |
| EU AI Act | Article 99 | COMPLIANT | ATTESTED |
| NIST AI RMF | GOVERN | NON-COMPLIANT | AUTOMATED |
| NIST AI RMF | MAP | NOT ASSESSED | — |
| NIST AI RMF | MEASURE | NON-COMPLIANT | AUTOMATED |
| NIST AI RMF | MANAGE | NON-COMPLIANT | AUTOMATED |
| NIST AI 600-1 | Hallucination Tracking | COMPLIANT | ATTESTED (Phase 7 + penalty-correction, combined) |
| NIST AI 600-1 | Source Data Lineage | COMPLIANT | ATTESTED |
| NIST AI 600-1 | Output Monitoring | COMPLIANT | ATTESTED |
| NIST AI 600-1 | Confabulation Prevention | PARTIAL | ATTESTED |
| ISO/IEC 42001 | Clause 6 | NON-COMPLIANT | AUTOMATED |
| ISO/IEC 42001 | Clause 8 | NON-COMPLIANT | AUTOMATED |
| ISO/IEC 42001 | Clause 9 | NOT ASSESSED | — |

**Final tally: 17 of 22 assessed, 10 of 22 with automated evidence, 5 confirmed genuinely empty.**

---

### Coverage Across the Five Demonstrations

| Demo | What was added | Coverage | Overall verdict |
|---|---|---|---|
| 1 | Phase 8 automated results, Articles 12 and 18 now correctly mapped | 10/22 | NOT APPROVED |
| 2 | + Article 27, manual intake | 11/22 | NOT APPROVED |
| 3 | + NIST AI 600-1, real Phase 7 RAG findings | 15/22 | NOT APPROVED |
| 4 | + Article 15, Phase 4 and Phase 6, combined | 16/22 | NOT APPROVED |
| 5 | + Article 99, cross-verified penalty structure and correction event | 17/22 | NOT APPROVED |

The overall verdict never changed across all five demonstrations. Adding
real, even genuinely positive, evidence elsewhere (Article 12's
COMPLIANT status, NIST AI 600-1's three COMPLIANT results, Article 99's
COMPLIANT status) never overrides TalentMatch AI's actual, unresolved
Article 10 and Article 14 failures. A Conformity Report's headline
verdict is governed by its worst finding, not its best one.

---

### How the Two Safety Rules Work

**Rule 1, AUTOMATED evidence cannot be downgraded.** When `intake_item()`
is called on an obligation that already carries AUTOMATED evidence, the
call is refused outright, with a visible message stating why, rather
than failing silently. This was tested twice in this deliverable: once
deliberately (Phase 5's bias finding attempting to overwrite Article
10's Phase 8 evidence), and once incidentally (the same Phase 3 intake
call also collided with NIST RMF's MEASURE function, since Phase 8's
bias finding had already populated it through the cross-framework
mapping). Both were correctly refused.

**Rule 2, a second ATTESTED finding is appended, not substituted.** When
an obligation already has one manual finding and a second genuine
finding arrives, both are kept, clearly separated, with both sources
cited. This was tested with Phase 4 and Phase 6's separate Article 15
findings, and again with the penalty-correction event appending to NIST
AI 600-1's existing Phase 7 Hallucination Tracking finding. In both
cases, neither finding erased the other.

---

### Why This Level of Verification Matters

Every gap in this report has now been checked twice: once when it was
first left empty, and once when the question "is this really empty?"
was asked directly. Two of the six remaining gaps turned out to have
real evidence sitting unused in code that already existed. The other
five were confirmed, not assumed, to be genuinely empty. A Conformity
Report's credibility rests on this distinction: a regulator or client
reading NOT ASSESSED needs to trust that it means "no evidence exists,"
not "no one checked."

---

### Recommendation
Proceed to the cross-border framework deliverable. The verification
discipline built here, checking every NOT ASSESSED gap against real
project history before accepting it as genuinely empty, should carry
forward as the standard practice for every future deliverable, not a
one-time exercise specific to this report.